# Real-time Object Detection + LCOD Benchmark

Notebook tương tác để so sánh object detection theo video hoặc webcam. Workflow chuẩn:
**Run All → preflight checkpoint → chọn model và input → Play → theo dõi performance**.

- Closed-set: YOLO26l, YOLO26x, RF-DETR-L, RF-DETR-XL.
- Open-vocabulary / grounding: Grounding DINO, YOLO-World, YOLOE, Grounding DINO 1.5 Edge, LocateAnything-3B.
- Toàn bộ cache, checkpoint, file tạm và session metrics được lưu dưới `testmodel/`.


## Session 1 — Workspace và đường dẫn dự án

Xác định thư mục `testmodel` dù notebook chạy từ chính thư mục này hoặc thư mục cha. Session tạo các thư mục chuẩn cho cache model, video, log và kết quả session; đồng thời đọc cấu hình `.env*` mà không in secret ra output.


In [1]:
from __future__ import annotations
import os, sys
from pathlib import Path
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, HERE / 'testmodel') if (p / 'benchmark_object_detection.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Run notebook from project directory or parent')
CACHE = ROOT / 'cache'
HF = CACHE / 'huggingface'
MODELS = CACHE / 'models'
TMP = CACHE / 'tmp'
LOGS = ROOT / 'logs'
VIDEOS = ROOT / 'data' / 'videos'
SESSIONS = ROOT / 'outputs' / 'live_sessions'
for directory in (CACHE, HF, MODELS, TMP, LOGS, VIDEOS, SESSIONS):
    directory.mkdir(parents=True, exist_ok=True)

def read_env(path):
    out = {}
    if path.exists():
        for raw in path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if line and (not line.startswith('#')) and ('=' in line):
                key, value = line.split('=', 1)
                out[key.strip()] = value.strip().strip('"').strip("'")
    return out
ENVS = [read_env(ROOT / name) for name in ('.env.local', '.env.gdino15-edge', '.env.locateanything', '.env.main')]

def env_value(key, default=''):
    return os.environ.get(key) or next((env[key] for env in ENVS if env.get(key)), default)
print('Project root:', ROOT)


Project root: C:\Users\DELL\Desktop\VLM\testmodel


## Session 2 — Cache và dependency preflight

Chuyển toàn bộ cache của Hugging Face, PyTorch, Ultralytics và temporary files vào `testmodel/cache`. Session chỉ **kiểm tra** thư viện đã được chuẩn bị trước, tuyệt đối không `pip install` trong Run All.


In [2]:
import importlib.util
os.environ.update({'HF_HOME': str(HF), 'HUGGINGFACE_HUB_CACHE': str(HF / 'hub'), 'TRANSFORMERS_CACHE': str(HF / 'hub'), 'TORCH_HOME': str(CACHE / 'torch'), 'ULTRALYTICS_HOME': str(CACHE / 'ultralytics'), 'XDG_CACHE_HOME': str(CACHE / 'xdg'), 'TEMP': str(TMP), 'TMP': str(TMP), 'TMPDIR': str(TMP), 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TQDM_DISABLE': '1'})
required = ('torch', 'torchvision', 'ultralytics', 'transformers', 'huggingface_hub', 'ipywidgets', 'cv2', 'PIL', 'numpy', 'rfdetr', 'rfdetr_plus', 'dds_cloudapi_sdk')
missing = [module for module in required if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing))
import transformers
if tuple(map(int, transformers.__version__.split('.')[:2])) < (5, 1):
    raise RuntimeError('Main env requires Transformers >=5.1; LocateAnything uses a separate env')
print('Transformers:', transformers.__version__)
print('Local cache root:', CACHE)


Transformers: 5.17.0
Local cache root: C:\Users\DELL\Desktop\VLM\testmodel\cache


## Session 3 — Runtime, logging và GPU

Nạp các thư viện runtime, chọn CUDA nếu khả dụng, thiết lập dtype phù hợp và khai báo định dạng video được hỗ trợ. Output cho biết thiết bị đang thực thi để giải thích performance sau này.


In [3]:
import asyncio, concurrent.futures, csv, json, logging, time
from dataclasses import dataclass
import cv2, ipywidgets as widgets, numpy as np, torch
from IPython.display import display
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
LOG = logging.getLogger('lcod-lab')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
VIDEO_SUFFIXES = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
print('Device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')


Device: cuda
GPU: NVIDIA RTX A3000 12GB Laptop GPU


## Session 4 — Model registry và cấu trúc kết quả

Khai báo đúng một registry cho tất cả model cùng metadata: loại adapter, checkpoint/repository, và yêu cầu prompt. `Det` và `Result` là format chung cho mọi adapter, giúp UI và benchmark không phụ thuộc backend.


In [4]:
def managed_weight(env_key, default):
    """Keep prompted-YOLO weights inside testmodel/cache/models."""
    return MODELS / Path(env_value(env_key, default)).name
CONFIG = {'yolo26l': {'name': 'YOLO26l', 'type': 'yolo', 'weight': MODELS / 'yolo26l.pt', 'prompted': False}, 'yolo26x': {'name': 'YOLO26x', 'type': 'yolo', 'weight': MODELS / 'yolo26x.pt', 'prompted': False}, 'rfdetr_l': {'name': 'RF-DETR-L', 'type': 'rfdetr', 'variant': 'large', 'prompted': False}, 'rfdetr_xl': {'name': 'RF-DETR-XL', 'type': 'rfdetr', 'variant': 'xlarge', 'prompted': False}, 'grounding_dino': {'name': 'Grounding DINO', 'type': 'grounding_dino', 'repo': env_value('GROUNDING_DINO_REPO', 'IDEA-Research/grounding-dino-base'), 'prompted': True}, 'yolo_world': {'name': 'YOLO-World', 'type': 'yolo_world', 'weight': managed_weight('YOLO_WORLD_WEIGHT', 'yolov8l-worldv2.pt'), 'prompted': True}, 'yoloe': {'name': 'YOLOE', 'type': 'yoloe', 'weight': managed_weight('YOLOE_WEIGHT', 'yoloe-26l-seg.pt'), 'prompted': True}, 'gdino15_edge': {'name': 'Grounding DINO 1.5 Edge', 'type': 'gdino15_edge', 'prompted': True, 'remote': True}, 'locateanything_3b': {'name': 'LocateAnything-3B', 'type': 'locateanything', 'model': env_value('LOCATEANYTHING_MODEL', 'nvidia/LocateAnything-3B'), 'prompted': True}}

@dataclass
class Det:
    label: str
    score: float
    x1: float
    y1: float
    x2: float
    y2: float

@dataclass
class Result:
    detections: list[Det]
    ms: float


## Session 5 — Video metadata và visualization

Quét video đã có sẵn trong `data/videos`, đọc metadata để hiển thị rõ độ phân giải/thời lượng, và vẽ detection lên frame hiển thị. Session này không tải thêm video và không ghi đè video nguồn.


In [5]:
def videos():
    return sorted((p for p in VIDEOS.rglob('*') if p.suffix.lower() in VIDEO_SUFFIXES))

def video_text(path):
    c = cv2.VideoCapture(str(path))
    fps = c.get(cv2.CAP_PROP_FPS) or 0
    n = int(c.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w = int(c.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(c.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    c.release()
    return f'{path.name} | {w}x{h} | {(n / fps if fps else 0):.1f}s'

def paint(frame, dets):
    out = frame.copy()
    for d in dets:
        x1, y1, x2, y2 = map(int, (d.x1, d.y1, d.x2, d.y2))
        cv2.rectangle(out, (x1, y1), (x2, y2), (45, 220, 120), 2)
        score = '' if np.isnan(d.score) else f' {d.score:.2f}'
        cv2.putText(out, f'{d.label}{score}', (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (45, 220, 120), 2)
    return out


## Session 6 — Model adapters

Import adapter theo từng backend. Adapter chịu trách nhiệm download/load checkpoint đúng cache dự án, inference và chuyển output về `Result` thống nhất. LocateAnything chạy trong interpreter riêng để không xung đột Transformers của môi trường chính.


In [6]:
from lcod_adapters import Adapter, YOLOAdapter, RFDETRAdapter, GroundingDinoAdapter, YOLOWorldAdapter, YOLOEAdapter, GroundingDino15EdgeAdapter, LocateAnythingAdapter
CTX = {'ROOT': ROOT, 'HF': HF, 'DEVICE': DEVICE, 'Det': Det, 'Result': Result, 'env_value': env_value}


## Session 7 — Adapter registry và model lifecycle

`Manager` chỉ giữ một model đang hoạt động để tránh chiếm VRAM không cần thiết. Chuyển model sẽ unload model trước, sau đó load checkpoint mới; đây là lý do UI phản hồi rõ trạng thái loading trước khi Play.


In [7]:
ADAPTERS = {'yolo26l': YOLOAdapter, 'yolo26x': YOLOAdapter, 'rfdetr_l': RFDETRAdapter, 'rfdetr_xl': RFDETRAdapter, 'grounding_dino': GroundingDinoAdapter, 'yolo_world': YOLOWorldAdapter, 'yoloe': YOLOEAdapter, 'gdino15_edge': GroundingDino15EdgeAdapter, 'locateanything_3b': LocateAnythingAdapter}

class Manager:

    def __init__(self):
        self.key = None
        self.a = None

    def load(self, key, progress=None):
        if self.key == key and self.a is not None:
            return
        self.unload()
        self.a = ADAPTERS[key](key, CONFIG, CTX)
        self.a.prepare(progress or Progress())
        self.key = key

    def predict(self, key, frame, conf, prompt=''):
        self.load(key)
        return self.a.predict(frame, conf, prompt)

    def unload(self):
        if self.a is not None:
            self.a.unload()
        self.key = self.a = None
MANAGER = Manager()


## Session 8 — Checkpoint preflight và smoke test

Tải (nếu chưa có), load và chạy một frame smoke-test cho từng model. Thanh tiến trình hiển thị phần trăm, model hiện tại, thời gian đã chạy và ETA. Báo cáo checkpoint được ghi vào `cache/model_preflight.json` để lần Run All sau có thể đối chiếu.


In [8]:
class Progress:
    def __init__(self):
        self.bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description="Preflight:",
            layout=widgets.Layout(width="640px"),
        )
        self.step = widgets.HTML("<b>Waiting to prepare models.</b>")
        self.timing = widgets.HTML("Elapsed: 0.0s | ETA: calculating…")
        self.view = widgets.VBox(
            [
                widgets.HTML("<h3>Checkpoint preflight</h3>"),
                self.bar,
                self.step,
                self.timing,
            ]
        )
        self.started_at = time.perf_counter()

    def note(self, text):
        self.step.value = f"<code>{text}</code>"

    def update(self, completed, total, model_name):
        elapsed = time.perf_counter() - self.started_at
        self.bar.value = round(completed * 100 / total)
        if completed:
            eta = elapsed / completed * (total - completed)
            eta_text = f"{eta:.1f}s"
        else:
            eta_text = "calculating…"
        self.step.value = f"<b>Step {completed}/{total}:</b> {model_name}"
        self.timing.value = f"Elapsed: {elapsed:.1f}s | ETA: {eta_text}"


def smoke_frame():
    frame = np.full((640, 640, 3), 128, dtype=np.uint8)
    cv2.rectangle(frame, (120, 140), (300, 430), (210, 210, 210), -1)
    return frame


def prepare_all(progress):
    report = {}
    test_frame = smoke_frame()
    total = len(CONFIG)
    started_at = time.perf_counter()

    for position, key in enumerate(CONFIG, start=1):
        model = CONFIG[key]
        progress.update(position - 1, total, model["name"])
        progress.note(f"[{position}/{total}] Preparing checkpoint: {model['name']}")
        if key == 'gdino15_edge' and not env_value('DDS_API_TOKEN'):
            report[key] = {
                "state": "not_configured",
                "detail": "DDS_API_TOKEN is not configured; add it to .env.gdino15-edge to enable this cloud model",
                "prepare_s": 0.0,
            }
            progress.update(position, total, model["name"])
            continue
        model_started_at = time.perf_counter()
        try:
            MANAGER.load(key, progress)
            prompt = "person" if model.get("prompted") else ""
            result = MANAGER.predict(key, test_frame, 0.25, prompt)
            report[key] = {
                "state": "ready",
                "detail": "Checkpoint, load and smoke inference verified",
                "prepare_s": round(time.perf_counter() - model_started_at, 3),
                "smoke_ms": round(result.ms, 3),
            }
        except Exception as exc:
            report[key] = {
                "state": "failed",
                "detail": f"{type(exc).__name__}: {exc}",
                "prepare_s": round(time.perf_counter() - model_started_at, 3),
            }
        finally:
            MANAGER.unload()
            progress.update(position, total, model["name"])

    report_payload = {
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "duration_s": round(time.perf_counter() - started_at, 3),
        "models": report,
    }
    (CACHE / "model_preflight.json").write_text(
        json.dumps(report_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    progress.note("Preflight completed. Checkpoint report saved in cache/model_preflight.json")
    return report


## Session 9 — UI tương tác và live performance

UI hỗ trợ video cục bộ hoặc webcam. Với video, progress có phần trăm và ETA theo throughput thực tế; với webcam, UI báo thời gian session, inference latency trung bình và effective FPS. Stop an toàn: chỉ đóng input sau inference hiện tại, không ghi đè video nguồn.


In [9]:
EXECUTOR = concurrent.futures.ThreadPoolExecutor(max_workers=1)


class Lab:
    def __init__(self, ready, report):
        self.model = widgets.Dropdown(
            options=[(CONFIG[key]["name"], key) for key in ready],
            description="Model:",
            layout=widgets.Layout(width="420px"),
        )
        self.prompt = widgets.Text(
            value="person",
            description="Prompt:",
            layout=widgets.Layout(width="760px"),
        )
        self.source = widgets.ToggleButtons(
            options=[("Video file", "video"), ("Live webcam", "camera")],
            value="camera",
            description="Input:",
            tooltips=["Use a local file", "Use the selected local webcam"],
        )
        self.video = widgets.Dropdown(
            options=[(video_text(video), str(video)) for video in videos()],
            description="Video:",
            layout=widgets.Layout(width="760px"),
        )
        self.video_row = widgets.VBox([self.video])
        self.camera = widgets.BoundedIntText(value=0, min=0, max=9, description="Camera #:")
        self.camera_row = widgets.HBox([self.camera, widgets.HTML("<small>Webcam is used only while Play is active.</small>")])
        self.play = widgets.Button(description="Play", button_style="success", icon="play")
        self.stop = widgets.Button(description="Stop", button_style="warning", icon="stop", disabled=True)
        self.conf = widgets.FloatSlider(value=.35, min=.05, max=.95, step=.05, description="Confidence")
        self.image = widgets.Image(format="jpeg", layout=widgets.Layout(width="760px"))
        self.status = widgets.HTML("<b>Ready.</b>")
        self.progress = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            description="Session:",
            layout=widgets.Layout(width="760px"),
        )
        self.metrics = widgets.HTML()
        self.running = False
        self.rows = []
        self.capture = None
        self.stop_event = None

        self.model.observe(self.change_model, names="value")
        self.source.observe(self.change_source, names="value")
        self.play.on_click(lambda _: asyncio.create_task(self.run()))
        self.stop.on_click(lambda _: self.stop_session())
        self.change_model({"new": self.model.value})
        self.change_source({"new": self.source.value})
        ready_summary = "<br>".join(
            f"<b>{CONFIG[key]['name']}</b>: checkpoint verified in "
            f"{state.get('prepare_s', 0):.1f}s; smoke {state.get('smoke_ms', 0):.1f} ms"
            for key, state in report.items()
            if state["state"] == "ready"
        ) or "None"
        failed = "<br>".join(
            f"<b>{CONFIG[key]['name']}</b>: {state['detail']}"
            for key, state in report.items()
            if state["state"] == "failed"
        ) or "None"
        configuration = "<br>".join(
            f"<b>{CONFIG[key]['name']}</b>: {state['detail']}"
            for key, state in report.items()
            if state["state"] == "not_configured"
        )
        self.view = widgets.VBox(
            [
                widgets.HTML("<h3>Interactive Object Detection + LCOD Benchmark</h3>"),
                self.model,
                self.prompt,
                self.source,
                self.video_row,
                self.camera_row,
                widgets.HBox([self.play, self.stop]),
                self.conf,
                widgets.HTML("<b>Unavailable after preflight</b><br>" + failed),
                widgets.HTML("<b>Optional cloud model not configured</b><br>" + configuration) if configuration else widgets.HTML(),
                self.status,
                self.progress,
                self.metrics,
                self.image,
            ]
        )

    def show(self):
        display(self.view)

    def change_model(self, change):
        key = change.get("new")
        self.prompt.disabled = not CONFIG[key].get("prompted") if key else True

    def change_source(self, change):
        is_camera = change.get("new") == "camera"
        self.video_row.layout.display = "none" if is_camera else ""
        self.camera_row.layout.display = "" if is_camera else "none"
        if not self.running:
            self.status.value = (
                "<b>Live webcam selected.</b> Press Play to start."
                if is_camera
                else "<b>Local video selected.</b> Playback follows source FPS when possible."
            )

    def stop_session(self):
        if not self.running:
            return
        self.running = False
        if self.stop_event is not None:
            self.stop_event.set()
        if self.capture is not None:
            self.capture.release()
        self.stop.disabled = True
        self.status.value = "<b>Stop requested.</b> Closing input after the current inference finishes."

    def open_capture(self, is_camera):
        if not is_camera:
            path = Path(self.video.value)
            return cv2.VideoCapture(str(path)), path, path.stem
        index = int(self.camera.value)
        backend = getattr(cv2, "CAP_DSHOW", None)
        capture = cv2.VideoCapture(index, backend) if backend is not None else cv2.VideoCapture(index)
        if not capture.isOpened() and backend is not None:
            capture.release()
            capture = cv2.VideoCapture(index)
        return capture, None, f"webcam_{index}"

    async def wait_for_video_clock(self, deadline):
        delay = deadline - time.perf_counter()
        if delay <= 0:
            await asyncio.sleep(0)
            return self.running
        try:
            await asyncio.wait_for(self.stop_event.wait(), timeout=delay)
        except asyncio.TimeoutError:
            return self.running
        return False

    async def run(self):
        if self.running or not self.model.value:
            return
        if self.source.value == "video" and not self.video.value:
            self.status.value = "<b style='color:#b91c1c'>Select a video first.</b>"
            return

        key = self.model.value
        query = self.prompt.value.strip() if CONFIG[key].get("prompted") else ""
        if CONFIG[key].get("prompted") and not query:
            self.status.value = "<b style='color:#b91c1c'>Prompt required.</b>"
            return

        self.running = True
        self.stop_event = asyncio.Event()
        self.play.disabled = True
        self.stop.disabled = False
        for control in (self.model, self.prompt, self.source, self.video, self.camera, self.conf):
            control.disabled = True
        self.rows = []
        loop = asyncio.get_running_loop()
        capture = None
        try:
            self.status.value = f"<b>Loading {CONFIG[key]['name']}...</b>"
            await loop.run_in_executor(EXECUTOR, MANAGER.load, key)
            if not self.running:
                return

            is_camera = self.source.value == "camera"
            capture, path, source = self.open_capture(is_camera)
            self.capture = capture
            if not capture.isOpened():
                raise RuntimeError(f"Cannot open {'webcam ' + str(self.camera.value) if is_camera else path}")

            source_fps = capture.get(cv2.CAP_PROP_FPS) or 0.0
            frame_period = 1 / source_fps if source_fps > 0 else 1 / 30
            total_frames = 0 if is_camera else int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
            frame_index = 0
            self.progress.max = max(total_frames, 1)
            self.progress.value = 0
            self.progress.bar_style = "info" if is_camera else ""
            self.progress.description = "Live:" if is_camera else "Video:"
            began = time.perf_counter()
            next_frame_at = began

            while self.running:
                if not is_camera and not await self.wait_for_video_clock(next_frame_at):
                    break
                ok, frame = capture.read()
                if not ok or not self.running:
                    break
                frame_index += 1
                if not is_camera:
                    self.progress.value = min(frame_index, self.progress.max)
                if not is_camera:
                    next_frame_at += frame_period

                result = await loop.run_in_executor(
                    EXECUTOR,
                    MANAGER.predict,
                    key,
                    frame,
                    self.conf.value,
                    query,
                )
                if not self.running:
                    break

                painted = paint(frame, result.detections)
                encoded, jpg = cv2.imencode(".jpg", painted)
                self.image.value = jpg.tobytes() if encoded else b""
                self.rows.append(
                    {
                        "frame": frame_index,
                        "objects": len(result.detections),
                        "inference_ms": round(result.ms, 3),
                        "model": key,
                        "prompt": query,
                        "input": source,
                    }
                )
                elapsed = time.perf_counter() - began
                average_ms = sum(row["inference_ms"] for row in self.rows) / len(self.rows)
                effective_fps = len(self.rows) / elapsed if elapsed else 0.0
                if is_camera:
                    input_detail = f"<b>Input:</b> Webcam {self.camera.value} | <b>Capture FPS:</b> {source_fps:.1f}" if source_fps else f"<b>Input:</b> Webcam {self.camera.value}"
                else:
                    input_detail = f"<b>Frame:</b> {frame_index}/{total_frames or '?'} | <b>Source FPS:</b> {source_fps:.1f}" if source_fps else f"<b>Frame:</b> {frame_index}/{total_frames or '?'}"
                if is_camera:
                    progress_detail = f"<b>Live frames:</b> {frame_index}"
                else:
                    percent = frame_index * 100 / total_frames if total_frames else 0.0
                    remaining = max(total_frames - frame_index, 0) / effective_fps if effective_fps and total_frames else 0.0
                    progress_detail = f"<b>Progress:</b> {percent:.1f}% | <b>ETA:</b> {remaining:.1f}s"
                self.metrics.value = (
                    f"{input_detail} | {progress_detail} | <b>Objects:</b> {len(result.detections)} | "
                    f"<b>Inference:</b> {result.ms:.1f} ms | "
                    f"<b>Average:</b> {average_ms:.1f} ms | "
                    f"<b>Effective:</b> {effective_fps:.1f} FPS | "
                    f"<b>Elapsed:</b> {elapsed:.1f}s"
                )

            stopped = not self.running
            if self.rows:
                output = SESSIONS / f"{int(time.time())}_{key}_{source}.csv"
                with output.open("w", newline="", encoding="utf-8") as file:
                    writer = csv.DictWriter(file, fieldnames=self.rows[0].keys())
                    writer.writeheader()
                    writer.writerows(self.rows)
                self.status.value = f"<b>{'Session stopped' if stopped else 'Session completed'}.</b> Metrics: {output}"
            elif stopped:
                self.status.value = "<b>Session stopped before any frame was processed.</b>"
            else:
                self.status.value = "<b style='color:#b91c1c'>Input ended without processed frames.</b>"
        except Exception as exc:
            LOG.exception("session failed")
            self.status.value = f"<b style='color:#b91c1c'>{type(exc).__name__}: {exc}</b>"
        finally:
            if capture is not None:
                capture.release()
            self.capture = None
            self.stop_event = None
            self.running = False
            self.play.disabled = False
            self.stop.disabled = True
            for control in (self.model, self.source, self.video, self.camera, self.conf):
                control.disabled = False
            self.change_model({"new": self.model.value})


## Session 10 — Launch gate

Hiển thị preflight trước, sau đó mở UI với các model đã sẵn sàng. Mặc định UI vẫn xuất hiện khi một số model phụ thuộc token/remote service chưa sẵn sàng; đặt `VLM_REQUIRE_ALL_MODELS_READY=1` nếu muốn bắt buộc toàn bộ model pass.


In [10]:
PROGRESS = Progress()
display(PROGRESS.view)

REPORT = prepare_all(PROGRESS)
READY_KEYS = [key for key in CONFIG if REPORT[key]["state"] == "ready"]
strict_mode = env_value("VLM_REQUIRE_ALL_MODELS_READY", "0") == "1"

if strict_mode and len(READY_KEYS) != len(CONFIG):
    raise RuntimeError("Strict preflight failed; inspect cache/model_preflight.json")
if not READY_KEYS:
    raise RuntimeError("No model passed preflight; inspect cache/model_preflight.json")

LAB = Lab(READY_KEYS, REPORT)
LAB.show()


[2026-09-27 16:45:41] [INFO] rf-detr - File C:\Users\DELL\.roboflow\models\rf-detr-large-2026.pth already exists with correct MD5 hash.
[2026-09-27 16:45:42] [INFO] rf-detr - File C:\Users\DELL\.roboflow\models\rf-detr-large-2026.pth already exists with correct MD5 hash.
[2026-09-27 16:45:43] [INFO] rf-detr - File C:\Users\DELL\.roboflow\models\rf-detr-xlarge.pth already exists with correct MD5 hash.
[2026-09-27 16:45:45] [INFO] rf-detr - File C:\Users\DELL\.roboflow\models\rf-detr-xlarge.pth already exists with correct MD5 hash.


2026-09-27 16:45:46,985 | INFO | HTTP Request: HEAD https://huggingface.co/IDEA-Research/grounding-dino-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-27 16:45:47,255 | INFO | HTTP Request: HEAD https://huggingface.co/IDEA-Research/grounding-dino-base/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-27 16:45:47,265 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/IDEA-Research/grounding-dino-base/12bdfa3120f3e7ec7b434d90674b3396eccf88eb/preprocessor_config.json?%2FIDEA-Research%2Fgrounding-dino-base%2Fresolve%2Fmain%2Fpreprocessor_config.json=&etag=%225cb45d963917ed130ce46a93204b349cbec21131%22 "HTTP/1.1 200 OK"
2026-09-27 16:45:47,558 | INFO | HTTP Request: HEAD https://huggingface.co/IDEA-Research/grounding-dino-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-27 16:45:47,819 | INFO | HTTP Request: HEAD https://huggingface.co/IDEA-Research/grounding-dino-base/resolve/main/prepr

## Session 11 — Cách sử dụng và đọc performance

1. Chạy **Run All** và chờ preflight hoàn tất.
2. Chọn model, video hoặc webcam; model open-vocabulary cần prompt.
3. Bấm **Play**. Video giữ nhịp FPS nguồn khi inference kịp; nếu model chậm hơn, playback sẽ chậm thay vì bỏ frame/tua nhanh.
4. Theo dõi progress, elapsed/ETA, inference latency, latency trung bình và effective FPS. Sau session, metrics từng frame được lưu trong `outputs/live_sessions`.

`Grounding DINO 1.5 Edge` cần `DDS_API_TOKEN` trong `.env.local`; latency bao gồm network/service time. `LocateAnything-3B` dùng interpreter riêng khai báo trong `.env.locateanything`.
